<div dir="rtl" style="text-align: right">

# לוח תשובות: תרגול שבוע 1

מחברת למתרגל, להצגה על הלוח. היא קוראת את גיליון התשובות כל כמה שניות ומציגה כמה זוגות הגישו כל תשובה בסבב הנוכחי. התשובות מעוגלות לשלוש ספרות משמעותיות, כך שתשובות זהות מתקבצות יחד.

1. מריצים את תא ההגדרות פעם אחת.
2. לפני כל סבב משנים את `ROUND` ומריצים את תא התצוגה. עוצרים אותו בכפתור העצירה.
3. בזמן הצגת הפתרון כותבים את התשובה הנכונה ב-`ANSWER` ומריצים שוב, והיא מסומנת בירוק. התשובות לא שמורות במחברת, כי הריפו ציבורי.

</div>

In [ ]:
import io
import time
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output

# Published-to-web CSV of the responses sheet
CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTw80Yg-IKnLu2onKXwB1RzjtP2k0tc0VzjA0wCs4P6UTm3gI5EVkPE8hk2pRopU0YUWsxIcv1Yd7nB/pub?gid=1276875717&single=true&output=csv"
# If updates show up minutes late, share the sheet as "Anyone with the link: Viewer" and use instead:
# CSV_URL = "https://docs.google.com/spreadsheets/d/<SHEET_ID>/gviz/tq?tqx=out:csv&gid=1276875717"

def label(value):
    """Group answers: numbers to 3 significant figures, text stripped."""
    try:
        x = float(str(value).replace(",", ""))
    except ValueError:
        return str(value).strip()
    if abs(x) >= 1000:
        return f"{x:,.0f}"
    return f"{x:.3g}"


def load(round_name):
    sep = "&" if "?" in CSV_URL else "?"
    raw = urllib.request.urlopen(f"{CSV_URL}{sep}nocache={time.time():.0f}", timeout=10).read()
    df = pd.read_csv(io.BytesIO(raw), dtype=str)
    cols = {c.strip().lower(): c for c in df.columns}
    rcol = cols.get("round", df.columns[1])
    acol = cols.get("answer", df.columns[2])
    rows = df[df[rcol].fillna("").str.strip().str.lower() == round_name]
    return rows[acol].dropna()


def show(round_name, answer=None, top=8):
    answers = load(round_name)
    counts = answers.map(label).value_counts()
    if len(counts) > top:
        counts = pd.concat([counts.iloc[:top], pd.Series({"other": counts.iloc[top:].sum()})])
    counts = counts.iloc[::-1]
    right = None if answer is None else label(answer)
    colors = ["#2e9e5b" if k == right else "#4a7bb7" for k in counts.index]

    fig, ax = plt.subplots(figsize=(10, 0.6 * max(len(counts), 3) + 1.2))
    ax.barh(counts.index.astype(str), counts.values, color=colors)
    for y, v in enumerate(counts.values):
        ax.text(v, y, f" {v}", va="center", fontsize=14)
    ax.set_title(f"{round_name}: {len(answers)} submissions", fontsize=16)
    ax.tick_params(axis="y", labelsize=15)
    ax.set_xticks([])
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
ROUND = "gimli"        # gimli / dhahran / mars / ariane / signal
ANSWER = None          # type the correct answer here when you reveal it
REFRESH_S = 5

try:
    while True:
        clear_output(wait=True)
        try:
            show(ROUND, ANSWER)
        except Exception as e:
            print("Could not read the sheet:", e)
        time.sleep(REFRESH_S)
except KeyboardInterrupt:
    pass